In [ ]:
# Dicas para executar notebooks no Google Colab:
# `pip install eegdash`
# Configura exibição gráfica inline no Jupyter notebook
%matplotlib inline

# Enviar gravações do EEGDash para um paradigma do MOABB

Carregue imaginação motora real com o EEGDash, exponha essas mesmas gravações MNE
através da interface de dataset do MOABB e avalie as épocas resultantes do MOABB.
Este pequeno adaptador demonstra a interoperabilidade de sinais e rótulos; não é
um benchmark completo do MOABB. Instale a dependência opcional ``moabb``.

Use [NEMAR nm000135](https://nemar.org/dataset/nm000135) (BNCI2014-004),
sujeito 1, sessões ``0train`` e ``1train``, execução 0: aproximadamente 11 MB.
Defina ``EEGDASH_CACHE_DIR`` para reutilizar o primeiro download. A versão já está
processada; o MOABB aplica o filtro de análise explicitamente selecionado de 8–30 Hz.
Dependências, downloads ou rótulos ausentes geram erros em vez de produzir
resultados substitutos.

Pré-requisitos: a divisão de sessão real do tutorial 52 e familiaridade com MNE Raw
e Epochs. Instale o EEGDash e o pacote opcional MOABB em um ambiente Python compatível.
Uma importação ausente do MOABB deve ser resolvida antes da execução;
o notebook tem um caminho de aquisição e um caminho de avaliação medida.
A saída útil é um objeto de épocas produzido pelo MOABB com a identidade da sessão retida,
seguido por predições para a sessão reservada.


## 1. Carregar e inspecionar a fonte de sinais do EEGDash



In [ ]:
# Importa utilitários de sistema operacional e manipulação de diretórios
import os
from pathlib import Path

# Importa bibliotecas para gráficos, computação matricial e manipulação tabular
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
# Importa classes de dataset base e paradigma de imaginação motora do MOABB
from moabb.datasets.base import BaseDataset
from moabb.paradigms import LeftRightImagery
# Importa regressor logístico, matriz de confusão e métricas do scikit-learn
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, balanced_accuracy_score
# Importa pipeline e padronizador de escala
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

# Importa classes de dataset e cálculo de variância do sinal do EEGDash
from eegdash import EEGDashDataset
from eegdash.features import signal_variance

# Carrega a fonte de dados do sujeito 1 em duas sessões de imaginação motora
source = EEGDashDataset(
    cache_dir=Path(os.environ.get("EEGDASH_CACHE_DIR", ".eegdash_cache")),
    dataset="nm000135",
    subject="1",
    session=["0train", "1train"],
    run="0",
    task="imagery",
    n_jobs=1,
)
# Valida que exatamente duas sessões foram carregadas
assert len(source.datasets) == 2
# Exibe resumo dos metadados das gravações
print(source.description[["subject", "session", "run"]])
# Itera pelas gravações para verificar canais, taxa de amostragem e rótulos de eventos
for recording in source.datasets:
    raw = recording.raw
    print(raw.ch_names, raw.info["sfreq"], np.unique(raw.annotations.description))
    # Assegura que as anotações contêm os eventos esperados de mão esquerda e mão direita
    assert {"left_hand", "right_hand"}.issubset(raw.annotations.description)

## 2. Adaptar as gravações já carregadas ao contrato de dataset público do MOABB
O MOABB requer dicionários aninhados de sujeito/sessão/execução. Mantenha os nomes
genuínos de sessão e execução do BIDS. Retorne cópias porque o processamento do paradigma pode alterar
os objetos MNE. Códigos inteiros de eventos apenas codificam os nomes de anotação observados.
Este adaptador é exigido pela interface externa de datasets do MOABB; ele não é
uma segunda abstração de aquisição ou extração de características. Ele implementa
os dois métodos abstratos exigidos por BaseDataset.
``_get_single_subject_data`` é o hook provedor de dados do MOABB, apesar de seu sublinhado inicial.
``data_path`` intencionalmente não possui um segundo mecanismo de download:
o MOABB recebe os objetos do EEGDash que já adquirimos. A verificação de igualdade
abaixo confirma que a cópia preserva as amostras reais antes da filtragem.

``interval=[0, 3]`` declara nosso intervalo de análise relativo a cada pista existente.
``sessions_per_subject=2`` descreve a seleção real de duas sessões,
não uma solicitação para o MOABB sintetizar ou buscar sessões adicionais.



In [ ]:
# Classe adaptadora que conecta o dataset carregado pelo EEGDash à interface BaseDataset do MOABB
class EEGDashImagery(BaseDataset):
    def __init__(self, recordings):
        self.recordings = recordings
        super().__init__(
            subjects=[1],
            sessions_per_subject=2,
            events={"left_hand": 1, "right_hand": 2},
            code="EEGDashImagery",
            interval=[0, 3],
            paradigm="imagery",
        )

    # Método hook obrigatório pelo MOABB para estruturar os dados do sujeito em dicionários
    def _get_single_subject_data(self, subject):
        assert subject == 1
        return {
            str(recording.description["session"]): {
                str(recording.description["run"]): recording.raw.copy().load_data()
            }
            for recording in self.recordings.datasets
        }

    # Substitui a função de download do MOABB, pois os dados já foram baixados pelo EEGDash
    def data_path(
        self, subject, path=None, force_update=False, update_path=None, verbose=None
    ):
        raise NotImplementedError("EEGDash owns acquisition; use the loaded recordings")


# Instancia a classe adaptadora passando as gravações do EEGDash
adapter = EEGDashImagery(source)
# Verifica a transferência real do sinal antes de o MOABB filtrar ou reescalonar qualquer dado.
handed_off = adapter._get_single_subject_data(1)
for recording in source.datasets:
    raw = handed_off[str(recording.description["session"])][
        str(recording.description["run"])
    ]
    # Valida que as matrizes de amostras transferidas são exatamente idênticas às originais
    np.testing.assert_array_equal(raw.get_data(), recording.raw.get_data())

## 3. Permitir que o MOABB crie épocas rotuladas a partir dessas gravações do EEGDash
MNE Epochs retêm unidades em volts. O intervalo explícito começa na pista existente;
nenhuma correção de latência extra é aplicada. A ordem dos canais é fixada pelo nome.



In [ ]:
# Configura o paradigma do MOABB de imaginação motora esquerda/direita com filtro passa-banda de 8 a 30 Hz
paradigm = LeftRightImagery(
    fmin=8, fmax=30, tmin=0, tmax=3, channels=["C3", "Cz", "C4"]
)

Retornar Epochs mantém explícitas as unidades em volts do MNE. Com extremidades inclusivas,
0–3 segundos a 250 Hz resultam em 751 amostras, diferentemente das janelas fixas de 750 amostras
do Braindecode no tutorial 52. Esse ponto final e o filtro extra de 8–30 Hz
significam que as pontuações entre as duas páginas não representam uma comparação pareada de pipelines.
O MOABB fornece strings com nomes de classes em y e identidades de sessão nos metadados.



In [ ]:
# Processa os dados através do paradigma do MOABB retornando os objetos Epochs do MNE
epochs, y, metadata = paradigm.get_data(adapter, subjects=[1], return_epochs=True)
# Obtém a matriz de dados brutos das épocas (n_ensaios, n_canais, n_amostras)
X = epochs.get_data()
# Assegura alinhamento dimensional entre dados, classes e metadados
assert len(X) == len(y) == len(metadata)
# Assegura finitude numérica e consistência das classes rotuladas
assert np.isfinite(X).all() and set(y) == {"left_hand", "right_hand"}
# Confirma a presença das duas sessões esperadas nos metadados
assert set(metadata.session) == {"0train", "1train"}
# Imprime as dimensões do array, nomes dos canais e unidades
print("MOABB epochs:", X.shape, epochs.ch_names, "units: volts")
# Exibe tabela cruzada de distribuição de classes por sessão
print(pd.crosstab(metadata.session, y))

## 4. Ajustar apenas na primeira sessão e prever a segunda
A função ``signal_variance`` do EEGDash resume o sinal filtrado por banda por canal
em V²; aplicar o log fornece três características por ensaio. Chamar a função
pública diretamente aceita o array do MOABB sem outro adaptador de dataset.
Trata-se de uma transformação fixa por ensaio. O escalonamento e a classificação aprendem apenas do treino.



In [ ]:
# Calcula a variância de cada canal e aplica logaritmo natural com piso numérico de segurança
features = np.log(np.maximum(signal_variance(X), 1e-30))
# Define os índices de treino (sessão '0train') e de teste (sessão '1train')
train = np.flatnonzero(metadata.session.to_numpy() == "0train")
test = np.flatnonzero(metadata.session.to_numpy() == "1train")
# Assegura que treino e teste são estritamente disjuntos
assert set(train).isdisjoint(test)
# Assegura que ambas as classes estão presentes no treino e no teste
assert set(y[train]) == set(y[test]) == {"left_hand", "right_hand"}
# Cria o pipeline de aprendizagem com padronização e regressão logística
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
# Ajusta o modelo estritamente na sessão de treino '0train'
model.fit(features[train], y[train])
# Realiza predições para os ensaios da sessão retida '1train'
prediction = model.predict(features[test])
# Calcula e exibe a acurácia balanceada obtida na sessão retida
print(
    "Held-out session balanced accuracy:", balanced_accuracy_score(y[test], prediction)
)

## 5. Inspecionar predições a partir das épocas reais produzidas pelo MOABB



In [ ]:
# Plota a matriz de confusão normalizada para a sessão retida 1train
ConfusionMatrixDisplay.from_predictions(y[test], prediction, normalize="true")
plt.title("EEGDash → MOABB: subject 1, held-out session 1train")
# Exibe o gráfico da matriz de confusão
plt.show()
# Um benchmark completo adicionalmente precisa de mais participantes, um protocolo
# de avaliação pré-registrado e gerenciamento explícito de cache de resultados do MOABB.

## Inspecionar e estender a integração
O array resultante possui 240 ensaios, três canais motores e 751 amostras.
A variância em log reduz cada ensaio a três características; o StandardScaler então
usa apenas a sessão 0train. A matriz de confusão é normalizada por linha: entradas da diagonal
são as sensibilidades de mão esquerda e direita, cuja média é a acurácia balanceada.
Uma diagonal fraca é uma limitação medida do modelo, não uma falha de integração.

Para outro subconjunto de imaginação motora, primeiro verifique o vocabulário de anotações, o intervalo
da pista e os canais nomeados, e então atualize a declaração do adaptador para corresponder
às gravações consultadas. Antes de usar um avaliador do MOABB, configure também o armazenamento
de resultados dele e adicione participantes suficientes para a unidade de avaliação pretendida.
Esta página comprova a fronteira entre o sinal e o paradigma; ela não reproduz um
placar (*leaderboard*) publicado do MOABB.

